# Multimode mode-sum optimization

The same control/objective/solve/report pattern on a two-mode propagation problem.

In [ ]:
import Pkg
project_root = isfile("Project.toml") ? pwd() : dirname(pwd())
Pkg.activate(project_root; io = devnull)
cd(project_root)

using Base64
using FiberLab

function show_report(report)
    println("Standard figures are embedded in the notebook below.")
    report
end

In [ ]:
grid = Grid(nt = 1024, time_window_ps = 12.0, policy = :exact)
fiber = Fiber(regime = :multimode, preset = :two_mode_demo, length_m = 2.0, power_w = 0.2)

problem = fiber_problem(
    fiber;
    modes = 2,
    grid = grid,
    initial_modes = [1 / sqrt(2), 1im / sqrt(2)],
    dispersion = hcat(zeros(grid.nt), 1e-4 .* collect(range(-1.0, 1.0; length = grid.nt))),
    gamma_tensor = begin
        γ = zeros(Float64, 2, 2, 2, 2)
        γ[1, 1, 1, 1] = 1.0e-3
        γ[2, 2, 2, 2] = 0.8e-3
        γ[1, 1, 2, 2] = 0.15e-3
        γ[2, 2, 1, 1] = 0.12e-3
        γ
    end,
    raman_threshold_thz = -13.2,
)

basis = fourier_basis(problem, 8)
control = phase_control(problem; basis = basis, bounds = (-4.0, 4.0))
objective = mode_sum_objective(problem)

dimension(control), summarize(problem)

In [ ]:
result = solve(
    problem,
    control,
    objective;
    id = "multimode_demo",
    max_iter = 8,
    step_size = 1e-2,
    maturity = :supported,
)

report = standard_report(
    problem,
    result;
    output_dir = joinpath("examples", "outputs", "03_multimode_mode_sum"),
    tag = "multimode_demo",
    n_z_samples = 72,
)

display_report(report)
metrics(result)


In [ ]:
show_report(report)

<!-- fiberlab-standard-figures -->
## Standard figures

### Evolution comparison
![Evolution comparison](attachment:evolution_comparison.png)

### Phase profile
![Phase profile](attachment:phase_profile.png)

### Evolution
![Evolution](attachment:evolution.png)

### Phase diagnostic
![Phase diagnostic](attachment:phase_diagnostic.png)

### Unshaped evolution
![Unshaped evolution](attachment:evolution_unshaped.png)

